In [7]:
import duckdb
import pandas as pd

print("🦆 Extraindo Logs de Interação (Direto do arquivo base)...")
pasta_dados = "export/" 

# Busca simplificada, apenas no arquivo de logs
query_logs = f"""
    SELECT timecreated, username, eventname, action, 
           component, contextinstanceid, courseid
    FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE)
    WHERE username NOT IN ('0', '-1', '', 'nan') AND username IS NOT NULL
    ORDER BY CAST(timecreated AS BIGINT) ASC
"""
df_logs = duckdb.query(query_logs).df()

# Criamos a coluna 'section' artificialmente com valor nulo para não quebrar o transformador
df_logs['section'] = 'nan'

def analisar_dataset_moodle(df):
    print("📊 --- RADIOGRAFIA DO DATASET DO MOODLE --- 📊\n")
    
    # 1. Total de Eventos (Linhas de Log)
    total_eventos = len(df)
    print(f"🔹 Total de Interações (Logs): {total_eventos:,}".replace(',', '.'))
    
    # 2. Total de Alunos Únicos
    # Filtra possíveis usuários nulos ou do sistema ('0', '-1')
    alunos_validos = df[~df['username'].isin(['0', '-1', 'nan', 'None', ''])]
    total_alunos = alunos_validos['username'].nunique()
    print(f"👨‍🎓 Total de Alunos Únicos: {total_alunos:,}".replace(',', '.'))
    
    # 3. Total de Matérias (Courses)
    materias_validas = df[~df['courseid'].isin(['0', 'nan', 'None', ''])]
    total_materias = materias_validas['courseid'].nunique()
    print(f"📚 Total de Matérias (Cursos): {total_materias:,}".replace(',', '.'))
    
    # 4. Total de Seções
    secoes_validas = df[~df['section'].isin(['0', 'nan', 'None', ''])]
    total_secoes = secoes_validas['section'].nunique()
    print(f"🗂️ Total de Seções: {total_secoes:,}".replace(',', '.'))
    
    # 5. Total de Atividades Únicas (Context Instances)
    atividades_validas = df[~df['contextinstanceid'].isin(['0', 'nan', 'None', ''])]
    total_atividades = atividades_validas['contextinstanceid'].nunique()
    print(f"📝 Total de Atividades: {total_atividades:,}".replace(',', '.'))
    
    print("\n🏆 --- TOP 5 EVENTOS MAIS COMUNS (O que os alunos mais fazem?) --- 🏆")
    top_eventos = df['eventname'].value_counts().head(5)
    for evento, qtd in top_eventos.items():
        print(f"   - {evento}: {qtd:,}".replace(',', '.'))
        
    print("\n🧩 --- TOP 5 COMPONENTES (Onde eles mais clicam?) --- 🧩")
    top_componentes = df['component'].value_counts().head(5)
    for comp, qtd in top_componentes.items():
        print(f"   - {comp.replace('mod_', '')}: {qtd:,}".replace(',', '.'))

# AQUI ESTÁ A CORREÇÃO: Passando a variável df_logs correta!
analisar_dataset_moodle(df_logs)

🦆 Extraindo Logs de Interação (Direto do arquivo base)...
📊 --- RADIOGRAFIA DO DATASET DO MOODLE --- 📊

🔹 Total de Interações (Logs): 2.391.762
👨‍🎓 Total de Alunos Únicos: 2.168
📚 Total de Matérias (Cursos): 5.202
🗂️ Total de Seções: 0
📝 Total de Atividades: 17.707

🏆 --- TOP 5 EVENTOS MAIS COMUNS (O que os alunos mais fazem?) --- 🏆
   - \core\event\message_sent: 544.059
   - \core\event\webservice_function_called: 257.108
   - \core\event\course_viewed: 251.592
   - \mod_forum\event\discussion_viewed: 152.132
   - \core\event\message_viewed: 150.878

🧩 --- TOP 5 COMPONENTES (Onde eles mais clicam?) --- 🧩
   - core: 1.675.033
   - forum: 322.043
   - book: 119.033
   - quiz: 66.706
   - page: 45.813


In [8]:
import duckdb

query_cursos_reais = fquery_logs = f"""
        SELECT logs.timecreated, logs.username, logs.eventname, logs.action, 
            logs.component, logs.contextinstanceid, logs.courseid, cm.section AS section
    FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE) AS logs
    LEFT JOIN read_csv_auto('{pasta_dados}mdl_course_modules.csv', ALL_VARCHAR=TRUE) AS cm
        ON logs.contextinstanceid = cm.id
    WHERE logs.username NOT IN ('0', '-1', '', 'nan') AND logs.username IS NOT NULL
    AND courseid IN ('10464')
    ORDER BY CAST(logs.timecreated AS BIGINT) ASC
"""
df_cursos = duckdb.query(query_cursos_reais).df()

(df_cursos).to_csv("cursos_reais.csv", index=False)

In [10]:
quantas_complecoes = f"""
    SELECT 
        COUNT(DISTINCT logs.id) as qtd_complecoes
    FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE) AS logs
    WHERE logs.username NOT IN ('0', '-1', '', 'nan') 
      AND logs.username = 'user7219802108604710913'
      AND logs.courseid = '10464'
      AND logs.eventname = '\\mod_quiz\\event\\attempt_submitted'
      AND logs.contextinstanceid = '54994'
    GROUP BY logs.courseid
"""

df_cursos = duckdb.query(quantas_complecoes).df()
display(df_cursos)

,qtd_complecoes
0,10
